# 왜 사용자가 준 문장이 저장소 프롬프트보다 결함을 더 많이 찾았나
### why_user_prompts_found_more

분석일: 2026-07-30 · 대상: `unity_local_mcp` v1.11.9 → v1.11.15

표는 전부 저장소의 `prompts/`와 `logs/`에서 다시
계산하고, 결함 목록만 사람이 분류한 값으로 들어간다(각 항목에 근거 문서를 달았다).

## 요약

| | 저장소 프롬프트 | 사용자 문장 |
|---|---:|---:|
| 고유 프롬프트 | 9개 | **3개** |
| 발견한 하네스 결함 | 5개 | **15개** |
| 프롬프트당 결함 | 0.56 | **5.0** |

원인은 표본 수가 아니다. **저장소 프롬프트는 추출기를 만든 사람이 썼다.** 같은 사람이 쓴 문장은 그 사람이 이미 생각해 둔 표현만 쓰므로, 어휘 공백을 원리적으로 밟지 못한다.

#### 코퍼스(Corpus, 말뭉치)
- 검증에 사용된 프롬프트 (요청 문장)들의 집합

## 0. 준비 — 저장소에서 데이터 읽기

의존성 없이 표준 라이브러리와 `IPython.display`만 쓴다. 노트북을 `docs/`에서 열어도
저장소 루트에서 열어도 동작한다.

In [ ]:
import json, os, re, sys, glob, collections
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "prompts").is_dir():
    ROOT = ROOT.parent          # docs/ 에서 열었을 때
assert (ROOT / "prompts").is_dir(), f"저장소 루트를 찾지 못했다: {Path.cwd()}"
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from verification import VerificationSpec   # 추출기 본체
print("저장소:", ROOT)
print("프롬프트 파일:", len(glob.glob("prompts/*.txt")), "개")

: 

In [ ]:
from IPython.display import HTML, display

PALETTE = {"repo": "#8892a6", "user": "#2f6fdb", "accent": "#d9534f", "muted": "#cbd2dd"}

def bars(title, rows, unit="", width=680, colors=None, note=None):
    """수평 막대 하나짜리 SVG. rows = [(라벨, 값), ...]"""
    colors = colors or {}
    top = max(v for _, v in rows) or 1
    row_h, pad_l, pad_t = 34, 190, 40
    height = pad_t + row_h * len(rows) + (26 if note else 10)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
             f'font-family="system-ui,-apple-system,Segoe UI,sans-serif">']
    parts.append(f'<text x="0" y="22" font-size="15" font-weight="600">{title}</text>')
    for i, (label, value) in enumerate(rows):
        y = pad_t + i * row_h
        w = int((width - pad_l - 70) * value / top)
        c = colors.get(label, PALETTE["repo"])
        parts.append(f'<text x="{pad_l - 10}" y="{y + 16}" font-size="13" text-anchor="end">{label}</text>')
        parts.append(f'<rect x="{pad_l}" y="{y + 3}" width="{max(w, 2)}" height="18" rx="3" fill="{c}"/>')
        parts.append(f'<text x="{pad_l + max(w, 2) + 8}" y="{y + 17}" font-size="12.5" fill="#444">{value}{unit}</text>')
    if note:
        parts.append(f'<text x="0" y="{height - 8}" font-size="11.5" fill="#666">{note}</text>')
    parts.append("</svg>")
    display(HTML("".join(parts)))

def grid(title, cols, rows, cell, width=760, note=None):
    """표기 × 코퍼스 히트맵. cell(row, col) -> bool"""
    cw, ch, pad_l, pad_t = 90, 30, 210, 62
    height = pad_t + ch * len(rows) + (24 if note else 8)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
             f'font-family="system-ui,-apple-system,Segoe UI,sans-serif">']
    parts.append(f'<text x="0" y="22" font-size="15" font-weight="600">{title}</text>')
    for j, c in enumerate(cols):
        parts.append(f'<text x="{pad_l + j*cw + cw/2}" y="{pad_t - 10}" font-size="12.5" '
                     f'text-anchor="middle" font-weight="600">{c}</text>')
    for i, r in enumerate(rows):
        y = pad_t + i * ch
        parts.append(f'<text x="{pad_l - 12}" y="{y + 19}" font-size="12.5" text-anchor="end">{r}</text>')
        for j, c in enumerate(cols):
            on = cell(r, c)
            fill = PALETTE["user"] if (on and c.startswith("사용자")) else (PALETTE["repo"] if on else "#eef1f5")
            parts.append(f'<rect x="{pad_l + j*cw + 6}" y="{y + 4}" width="{cw - 12}" height="{ch - 9}" '
                         f'rx="3" fill="{fill}"/>')
            if on:
                parts.append(f'<text x="{pad_l + j*cw + cw/2}" y="{y + 20}" font-size="12" '
                             f'text-anchor="middle" fill="#fff">사용</text>')
    if note:
        parts.append(f'<text x="0" y="{height - 6}" font-size="11.5" fill="#666">{note}</text>')
    parts.append("</svg>")
    display(HTML("".join(parts)))

print("SVG 헬퍼 준비 완료 (외부 패키지 없음)")

## 1. 코퍼스는 생각보다 훨씬 작다

`prompts/`에는 파일이 40개 넘게 있지만, 씬 경로만 다른 복제본이 대부분이다. 실제로
서로 다른 문장은 12개뿐이고 그중 3개가 사용자가 준 것이다.

In [ ]:
USER_MARKS = ("3층짜리", "새씬을 열어서", "새 씬을 만들어서 2.5D")
norm = lambda t: re.sub(r"Assets/Scenes/\S+?\.unity", "<SCENE>", t).strip()

files = sorted(glob.glob("prompts/*.txt"))
unique = {}
for f in files:
    text = open(f, encoding="utf-8").read().strip()
    unique.setdefault(norm(text), text)

def source_of(text):
    return "user" if any(m in text for m in USER_MARKS) else "repo"

repo_prompts = [t for t in unique.values() if source_of(t) == "repo"]
user_prompts = [t for t in unique.values() if source_of(t) == "user"]

print(f"프롬프트 파일        {len(files)}")
print(f"씬 경로 제거 후 고유 {len(unique)}")
print(f"  ├ 저장소가 쓴 것   {len(repo_prompts)}")
print(f"  └ 사용자가 준 것   {len(user_prompts)}")

bars("코퍼스 압축", [
    ("prompts/ 파일", len(files)),
    ("고유 문장", len(unique)),
    ("저장소 작성", len(repo_prompts)),
    ("사용자 제공", len(user_prompts)),
], unit="개", colors={"사용자 제공": PALETTE["user"]},
     note="같은 문장을 씬 경로만 바꿔 재사용한 것이 대부분이다.")

## 2. 결정적 차이 — 같은 개념을 몇 가지로 쓰는가

추출기는 문자열을 본다. 그러므로 **같은 뜻을 다르게 적을 때마다 다른 코드 경로**다.
저장소 프롬프트 9개가 쓴 표기는 6종, 사용자 문장 3개가 쓴 표기는 12종이다.

In [ ]:
VARIANTS = {
    "A/D 이동": {"A/D": r"a\s*[/,·+]\s*d", "ad키": r"\bad\s*키",
                 "ad로": r"\bad\s*로", "A, D키": r"\bA,\s*D키"},
    "점프 키":  {"Space 점프": r"Space 점프", "space가": r"space\s*가", "space 키": r"space\s*키"},
    "부스트":   {"LeftShift": r"LeftShift", "좌쉬프트": r"좌쉬프트",
                 "쉬프트 키": r"쉬프트\s*키", "Shift 키": r"Shift\s*키"},
    "카메라":   {"따라오": r"따라오", "추적": r"추적"},
    "새 씬":    {"새 빈 씬": r"새 빈 씬", "새 씬": r"새 씬", "새씬": r"새씬"},
}
FLAT = {name: pat for group in VARIANTS.values() for name, pat in group.items()}

def used(prompts, pattern):
    return any(re.search(pattern, t, re.I) for t in prompts)

repo_used = {n for n, p in FLAT.items() if used(repo_prompts, p)}
user_used = {n for n, p in FLAT.items() if used(user_prompts, p)}

print(f"저장소 9개가 쓴 표기 {len(repo_used)}종: {sorted(repo_used)}")
print(f"사용자 3개가 쓴 표기 {len(user_used)}종: {sorted(user_used)}")
print(f"사용자만 쓴 표기      : {sorted(user_used - repo_used)}")

bars("같은 개념을 몇 가지 표현으로 쓰는가", [
    (f"저장소 프롬프트 {len(repo_prompts)}개", len(repo_used)),
    (f"사용자 문장 {len(user_prompts)}개", len(user_used)),
], unit="종", colors={f"사용자 문장 {len(user_prompts)}개": PALETTE["user"]},
     note="문장 수는 1/3인데 표기 다양성은 2배다.")

In [ ]:
grid("표기 × 코퍼스 — 파란 칸이 사용자만 밟은 경로",
     ["저장소", "사용자"],
     list(FLAT.keys()),
     lambda r, c: (r in repo_used) if c == "저장소" else (r in user_used),
     note="저장소는 개념마다 한 가지 표기만 쓴다. 추출기를 쓴 사람이 프롬프트도 썼기 때문이다.")

## 3. 그래서 결함이 어디서 나왔나

아래 목록은 이번 세션(v1.11.9 → v1.11.15)에 실제로 고친 하네스 결함이다. 각 항목의
근거는 `docs/` 아래 해당 버전 문서에 실측값과 함께 있다.

In [ ]:
DEFECTS = [
    # (출처, 종류, 요약, 문서)
    ("repo", "판정",  "격자 요청이 검사 0개로 verified — 공집합 성공 재발", "v1.11.13"),
    ("repo", "구조",  "호스트 파일 도구가 프로젝트 정체성 가드를 우회", "v1.11.13"),
    ("repo", "판정",  "빈 씬 템플릿에 카메라가 없어 전 검사 차단", "v1.11.12"),
    ("repo", "게이트", "미정의 Ground 태그 검사가 점프 요청에만 걸림", "v1.11.12"),
    ("repo", "판정",  "Play 종료 후 런타임 오류를 컴파일 오류로 계산 → rollback", "v1.11.12"),

    ("user", "어휘",  "ad키 미인식 → 하네스가 방향키로 측정", "v1.11.14"),
    ("user", "어휘",  "좌우 양방향 검사 누락", "v1.11.14"),
    ("user", "어휘",  "좌쉬프트 미인식 → 부스트 검사 없음", "v1.11.14"),
    ("user", "어휘",  "추적 미인식 → 카메라 검사 없음", "v1.11.14"),
    ("user", "안내",  "부스트 실패 수정 안내가 아예 없음", "v1.11.14"),
    ("user", "안내",  "점프 실패가 배치(천장)일 때 안내 없음", "v1.11.14"),
    ("user", "어휘",  "ad로 미인식 (조사만 다름)", "v1.11.14"),
    ("user", "구조",  "게이트와 측정이 어휘를 따로 보유 → 강제키 ≠ 측정키", "v1.11.14"),
    ("user", "어휘",  "새씬(붙여쓰기) 미인식 → 새 씬 정책 전체 꺼짐", "v1.11.14"),
    ("user", "어휘",  "게이트의 카메라 판정이 추적을 모름", "v1.11.14"),
    ("user", "구조",  "\\b 경계가 한글에서 깨짐 (D로의 d/로 사이에 경계 없음)", "v1.11.14"),
    ("user", "판정",  "플레이어에 붙은 시점 카메라가 추종 검사를 통과", "v1.11.15"),
    ("user", "게이트", "CameraController.cs를 입력 스크립트로 오인해 13회 차단", "v1.11.15"),
    ("user", "판정",  "부스트 상한 부재 — 0.5초 140유닛 대시가 통과", "v1.11.15"),
    ("user", "안내",  "장애물이 측정을 막을 때 안내 없음", "v1.11.15"),
]

by_source = collections.Counter(d[0] for d in DEFECTS)
print(f"저장소 프롬프트가 찾은 결함 {by_source['repo']}개")
print(f"사용자 문장이 찾은 결함   {by_source['user']}개")

bars("발견한 하네스 결함 수", [
    ("저장소 프롬프트", by_source["repo"]),
    ("사용자 문장", by_source["user"]),
], unit="개", colors={"사용자 문장": PALETTE["user"]})

bars("프롬프트 1개당 결함 (효율)", [
    ("저장소 프롬프트", round(by_source["repo"] / len(repo_prompts), 2)),
    ("사용자 문장", round(by_source["user"] / len(user_prompts), 2)),
], unit="개", colors={"사용자 문장": PALETTE["user"]},
     note="문장 하나가 만들어낸 결함 수. 9배 차이다.")

### 3.1 종류를 보면 이유가 드러난다

숫자보다 **어떤 종류**를 찾았는지가 중요하다.

In [ ]:
kinds = sorted({d[1] for d in DEFECTS})
table = {k: collections.Counter(d[0] for d in DEFECTS if d[1] == k) for k in kinds}

print(f"{'종류':6s} {'저장소':>6s} {'사용자':>6s}")
for k in kinds:
    print(f"{k:6s} {table[k]['repo']:6d} {table[k]['user']:6d}")

rows = []
for k in kinds:
    rows.append((f"{k} · 저장소", table[k]["repo"]))
    rows.append((f"{k} · 사용자", table[k]["user"]))
bars("결함 종류별 출처", rows, unit="개",
     colors={f"{k} · 사용자": PALETTE["user"] for k in kinds},
     note="어휘·안내 결함은 전부 사용자 문장에서만 나왔다.")

**어휘 결함 7건과 안내 결함 3건은 100% 사용자 문장에서 나왔다.** 저장소 프롬프트는
단 한 건도 찾지 못했다. 이것이 이 분석의 핵심이다.

반대로 저장소 프롬프트가 찾은 5건은 전부 **판정·구조** 결함이다. 새 요청 형태를
일부러 만들어 넣었을 때(카메라 추종, 격자, 레벨 데이터) 나온 것들이다.

## 4. 왜 이런 차이가 나는가

### 4.1 저장소 프롬프트는 추출기를 만든 사람이 썼다

가장 큰 이유다. `_AD_SCHEME`을 `a/d` 형태로 쓴 사람이 프롬프트도 `A/D 좌우 이동`이라고
쓴다. **자기가 구현한 표기로 자기 구현을 시험하는 것**이라, 통과는 보장되고 정보는
0이다. `ad키`·`ad로`·`좌쉬프트`·`추적`·`새씬`은 그 사람의 머릿속에 없던 표기다.

### 4.2 저장소 프롬프트는 통과를 목표로 다듬어졌다

프롬프트 파일들은 E2E를 **통과시키려고** 만들어졌다. 실패하면 프롬프트를 고쳤다.
그 과정 자체가 하네스의 빈틈을 피해 가는 문장을 만든다 — 살아남은 프롬프트는
정의상 결함을 밟지 않는 문장이다(생존자 편향).

### 4.3 사용자는 결과를 눈으로 본다

v1.11.15의 가장 큰 결함은 **문장이 아니라 사용자의 관찰**에서 나왔다.

> 카메라가 플레이어를 관찰하는 게 아니라 시점으로 되어 있어서

영수증은 그 실행을 `verified`로 적고 있었고, 플레이어에 붙은 카메라는 변위가 정확히
같아 추종 검사를 **완벽하게** 통과한다. 로그·영수증·단위 테스트 어디에도 이걸 드러낼
경로가 없었다. 화면을 본 사람만 알 수 있었다.

### 4.4 한 문장에 기능이 여럿 얽힌다

저장소 프롬프트는 대체로 한두 기능(이동+점프)만 검증했다. 사용자 문장은 이동·점프·
대시·카메라·층 구조를 한 번에 요구한다. 기능이 겹칠 때만 나오는 결함이 있다 —
`CameraController.cs` 오차단은 **카메라 스크립트와 입력 정책이 같은 실행에 있어야**
드러난다.

In [ ]:
# 4.4의 근거 — 문장별로 요청된 검사 개수
rows = []
for text in sorted(unique.values(), key=lambda t: len(VerificationSpec.from_request(t).requested_checks())):
    n = len(VerificationSpec.from_request(text).requested_checks())
    who = source_of(text)
    label = ("사용자: " if who == "user" else "저장소: ") + text[:24].replace("\n", " ")
    rows.append((label, n, who))

bars("문장 하나가 요구하는 검사 개수", [(l, n) for l, n, _ in rows], unit="개",
     colors={l: PALETTE["user"] for l, _, w in rows if w == "user"},
     note="사용자 문장은 기능이 얽혀 있어 상호작용 결함을 드러낸다.")

## 5. 그래서 무엇을 바꿔야 하나

### 5.1 프롬프트를 늘리지 말고 표기를 늘린다

파일 43개 중 고유 문장이 12개인 코퍼스는 **크기가 아니라 다양성이 문제**다. 같은 기능을
여러 표기로 쓴 변형을 넣는 편이 새 프롬프트 파일을 늘리는 것보다 낫다.

### 5.2 정적 추출을 먼저 돌린다 (E2E보다 300배 싸다)

E2E 한 번은 60~200초, 정적 추출은 1초 미만이다. §2의 표는 정적 추출만으로 만들었다.

In [ ]:
# 실제로 이렇게 쓴다 — 어떤 문장이든 넣어보면 무엇이 측정될지 즉시 나온다
probe = "wasd로 움직이고 시프트로 대시, 카메라가 플레이어를 쫓아다니게 해줘"
spec = VerificationSpec.from_request(probe)
print("요청 :", probe)
print("검사 :", spec.requested_checks())
print("이동키:", spec.move_right_key, "/", spec.move_left_key)
print("미매핑:", spec.unmapped_requirements())

### 5.3 사용자에게서 문장을 계속 받는다

이번 세션의 결론이자 가장 실행하기 쉬운 항목이다. 문장 3개가 결함 15개를 냈다.
**아직 밟지 않은 종류**일수록 값이 크다 — 적·충돌·점수처럼 이동 밖의 게임 로직,
또는 기존 씬을 수정하는 요청.

### 5.4 화면으로만 보이는 것을 위한 경로

§4.3의 카메라 결함은 계측으로 드러나지 않았다. 지금은 `camera_player_gap`이 영수증에
남아 같은 사례가 재발하면 사후 확인이 가능하다. 같은 질문을 다른 항목에도 해봐야 한다
— **"이 검사가 통과했는데도 화면이 이상할 수 있는 경우가 무엇인가?"**

## 6. 한계

- 결함 15 대 5는 **표본이 작다**(사용자 문장 3개). 비율보다 종류의 차이(§3.1)가 근거로
  더 단단하다.
- 결함의 출처 분류는 사람이 했다. 어떤 결함은 두 코퍼스 모두에서 나올 수 있었다.
- 저장소 프롬프트도 **새 요청 형태를 일부러 만들었을 때는** 결함을 찾았다(5건). 무용한
  것이 아니라, 이미 다루는 형태 안에서 무력한 것이다.
